In [ ]:
!pip install scikit-learn pandas numpy -q
print("Done")

Done


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

FOOD_ITEMS = [
    "Butter Chicken", "Masala Dosa", "Biryani", "Pizza Margherita",
    "Veg Fried Rice", "Paneer Tikka", "Sushi Platter", "Chocolate Ice Cream",
    "Samosa", "Pasta Arrabiata", "Garlic Naan", "Mango Lassi",
    "Chicken Burger", "Dal Makhani", "Gulab Jamun"
]

N_USERS = 300
N_ITEMS = len(FOOD_ITEMS)

# Generate sparse interaction matrix
ratings = []
for user_id in range(N_USERS):
    n_rated = np.random.randint(3, 10)
    items_rated = np.random.choice(N_ITEMS, size=n_rated, replace=False)
    for item_id in items_rated:
        rating = np.random.choice([3, 4, 4, 5, 5], p=[0.1, 0.2, 0.3, 0.2, 0.2])
        ratings.append({
            "user_id": f"U{user_id:04d}",
            "item_id": item_id,
            "food_name": FOOD_ITEMS[item_id],
            "rating": rating
        })

df = pd.DataFrame(ratings)
print(f"Total interactions: {len(df)}")
print(f"Users: {df['user_id'].nunique()} | Items: {df['item_id'].nunique()}")
print(f"\nSample:\n{df.head()}")

Total interactions: 1764
Users: 300 | Items: 15

Sample:
  user_id  item_id       food_name  rating
0   U0000        0  Butter Chicken       3
1   U0000        1     Masala Dosa       5
2   U0000        5    Paneer Tikka       5
3   U0000       14     Gulab Jamun       3
4   U0000       13     Dal Makhani       5


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Build user-item matrix
user_item_matrix = df.pivot_table(
    index="user_id", columns="food_name", values="rating"
).fillna(0)

print(f"Matrix shape: {user_item_matrix.shape}")

# Compute user-user similarity
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

def recommend(user_id, n=5):
    if user_id not in user_item_matrix.index:
        return []

    # Get top 10 similar users
    sim_scores = user_similarity_df[user_id].drop(user_id).nlargest(10)
    similar_users = sim_scores.index.tolist()

    # Items already rated by target user
    rated_items = set(
        user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index
    )

    # Score unrated items weighted by similarity
    scores = {}
    for sim_user, sim_score in zip(similar_users, sim_scores):
        rated = user_item_matrix.loc[sim_user]
        for item, rating in rated.items():
            if rating > 0 and item not in rated_items:
                scores[item] = scores.get(item, 0) + sim_score * rating

    recommendations = sorted(scores, key=scores.get, reverse=True)[:n]
    return recommendations


# Test
test_user = "U0001"
rated = df[df["user_id"] == test_user]["food_name"].tolist()
recs = recommend(test_user)

print(f"\nUser {test_user} previously ordered: {rated}")
print(f"Recommendations: {recs}")

Matrix shape: (300, 15)

User U0001 previously ordered: ['Masala Dosa', 'Dal Makhani', 'Samosa', 'Paneer Tikka', 'Veg Fried Rice']
Recommendations: ['Sushi Platter', 'Pizza Margherita', 'Biryani', 'Gulab Jamun', 'Chocolate Ice Cream']


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Item metadata
food_metadata = {
    "Butter Chicken": "chicken curry creamy indian spicy",
    "Masala Dosa": "south indian crispy crepe potato vegetarian",
    "Biryani": "rice chicken spicy indian aromatic",
    "Pizza Margherita": "italian pizza cheese tomato vegetarian",
    "Veg Fried Rice": "chinese rice vegetarian stir-fry",
    "Paneer Tikka": "indian vegetarian paneer grilled spicy",
    "Sushi Platter": "japanese seafood rice fresh healthy",
    "Chocolate Ice Cream": "dessert sweet cold chocolate creamy",
    "Samosa": "indian snack fried potato crispy vegetarian",
    "Pasta Arrabiata": "italian pasta spicy tomato vegetarian",
    "Garlic Naan": "indian bread garlic butter baked",
    "Mango Lassi": "indian drink mango yogurt sweet",
    "Chicken Burger": "fast food chicken grilled burger bun",
    "Dal Makhani": "indian lentil creamy vegetarian curry",
    "Gulab Jamun": "indian dessert sweet fried milk syrup",
}

# TF-IDF vectorization
items = list(food_metadata.keys())
descriptions = list(food_metadata.values())

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(descriptions)
item_similarity = cosine_similarity(tfidf_matrix)
item_sim_df = pd.DataFrame(item_similarity, index=items, columns=items)

def content_recommend(food_name, n=5):
    if food_name not in item_sim_df:
        return []
    scores = item_sim_df[food_name].drop(food_name).nlargest(n)
    return scores.index.tolist()

# Test
test_items = ["Biryani", "Pizza Margherita", "Chocolate Ice Cream"]
print("NourishNet AI - Content-Based Recommendations")
print("-" * 50)
for item in test_items:
    recs = content_recommend(item)
    print(f"Because you liked '{item}':")
    print(f"  → {recs}\n")

NourishNet AI - Content-Based Recommendations
--------------------------------------------------
Because you liked 'Biryani':
  → ['Butter Chicken', 'Paneer Tikka', 'Veg Fried Rice', 'Sushi Platter', 'Pasta Arrabiata']

Because you liked 'Pizza Margherita':
  → ['Pasta Arrabiata', 'Paneer Tikka', 'Dal Makhani', 'Samosa', 'Veg Fried Rice']

Because you liked 'Chocolate Ice Cream':
  → ['Gulab Jamun', 'Butter Chicken', 'Dal Makhani', 'Mango Lassi', 'Masala Dosa']



In [ ]:
import joblib
import os

os.makedirs("recommendation_models", exist_ok=True)

# Save all components
joblib.dump(user_item_matrix, "recommendation_models/user_item_matrix.pkl")
joblib.dump(user_similarity_df, "recommendation_models/user_similarity.pkl")
joblib.dump(item_sim_df, "recommendation_models/item_similarity.pkl")
joblib.dump(tfidf, "recommendation_models/tfidf_vectorizer.pkl")

print("All recommendation models saved.")

All recommendation models saved.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("recommendation_models", "zip", "recommendation_models")
files.download("recommendation_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>